In [13]:
import scanpy as sc
import numpy as np


import inspect
from functools import wraps
import numpy as np
import scanpy as sc

import inspect
from functools import wraps
import numpy as np
import scanpy as sc

def add_tracker(field_name="tracker"):
    def tracker(f):
        sig = None
        try:
            sig = inspect.signature(f)
        except (TypeError, ValueError):
            pass

        @wraps(f)
        def wrapper(*args, **kwds):
            adata = kwds.get("adata", None)
            if adata is None and len(args) > 0:
                adata = args[0]

            # If no AnnData-like object, pass through
            if adata is None or not hasattr(adata, "uns"):
                return f(*args, **kwds)

            # Only track if the function is in-place (or has no inplace param)
            if sig is not None and "inplace" in sig.parameters:
                inplace = kwds.get("inplace", sig.parameters["inplace"].default)
                if inplace is False:
                    return f(*args, **kwds)

            if field_name not in adata.uns:
                adata.uns[field_name] = np.array([], dtype="<U64")
            adata.uns[field_name] = np.append(adata.uns[field_name], [f.__name__])

            return f(*args, **kwds)

        return wrapper

    for name in dir(sc.pp):
        if name.startswith("_"):
            continue
        obj = getattr(sc.pp, name)
        if not callable(obj):
            continue
        try:
            sig = inspect.signature(obj)
        except (TypeError, ValueError):
            continue
        if "adata" in sig.parameters or "data" in sig.parameters:
            setattr(sc.pp, name, tracker(obj))


In [14]:
add_tracker()

a = sc.AnnData(np.random.randint(0, 100, size=(20, 5)))
assert "tracker" not in a.uns

sc.pp.normalize_total(a, inplace=True)
assert "tracker" in a.uns
assert (a.uns["tracker"] == np.array(["normalize_total"])).all()

sc.pp.log1p(a)  # log1p is inplace in scanpy
assert (a.uns["tracker"] == np.array(["normalize_total", "log1p"])).all()

sc.write("example_anndata.h5ad", a)
a2 = sc.read("example_anndata.h5ad")
assert "tracker" in a2.uns
assert (a2.uns["tracker"] == np.array(["normalize_total", "log1p"])).all()


ValueError: operands could not be broadcast together with shapes (6,) (2,) 

In [12]:
sc.__version__

/tmp/ipykernel_15732/4147423214.py:1: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  sc.__version__


'1.11.5'

# SIMPLE WITH @LOG

In [1]:
from anndata import AnnData
from datetime import datetime
from functools import wraps
from structlog import get_logger
from time import sleep
import uuid

logger = get_logger()

def logged(func):
    @wraps(func)
    def func_wrapper(*args, **kwargs):
        call_id = uuid.uuid4() # So we can always match call start with call end
        call_start_record = dict(call_id=call_id, called_func=func.__name__)
        if type(args[0]) is AnnData:
            call_start_record["adata_id"] = id(args[0])
        logger.msg("call", **call_start_record)

        t0 = datetime.now()
        output = func(*args, **kwargs)
        dt = datetime.now() - t0

        call_finish_record = dict(called_func=func.__name__, elapsed=dt)
        if type(output) is AnnData:
            call_finish_record["returned_adata_id"] = id(output)
        logger.msg("call_finish", **call_finish_record, call_id=call_id)
        return output
    return func_wrapper

# Usage

@logged
def foo(adata, x, copy=False):
    sleep(0.5)
    if copy: return adata.copy()

import scanpy as sc
pbmcs = sc.datasets.pbmc68k_reduced()

foo(pbmcs, 1)
# 2019-02-13 19:27.58 call                           adata_id=4937049368 call_id=UUID('82f3944c-08c1-470a-9d39-03dcabc091a2') called_func=foo
# 2019-02-13 19:27.58 call_finish                    call_id=UUID('82f3944c-08c1-470a-9d39-03dcabc091a2') called_func=foo elapsed=datetime.timedelta(microseconds=500777)
foo(pbmcs, 1, copy=True);
# 2019-02-13 19:28.02 call                           adata_id=4937049368 call_id=UUID('986f57e4-656a-41b1-9c7c-a7c5ad5b01fc') called_func=foo
# 2019-02-13 19:28.03 call_finish                    call_id=UUID('986f57e4-656a-41b1-9c7c-a7c5ad5b01fc') called_func=foo elapsed=datetime.timedelta(microseconds=505970) returned_adata_id=4940502352

2026-03-11 09:25:12 [info     ] call                           adata_id=137078809331408 call_id=UUID('6b1958c9-ef4d-4ded-8c6e-b4b4b889ffd5') called_func=foo
2026-03-11 09:25:12 [info     ] call_finish                    call_id=UUID('6b1958c9-ef4d-4ded-8c6e-b4b4b889ffd5') called_func=foo elapsed=datetime.timedelta(microseconds=500124)
2026-03-11 09:25:12 [info     ] call                           adata_id=137078809331408 call_id=UUID('4621a035-6f72-45ae-b969-cc6640ecd56e') called_func=foo
2026-03-11 09:25:13 [info     ] call_finish                    call_id=UUID('4621a035-6f72-45ae-b969-cc6640ecd56e') called_func=foo elapsed=datetime.timedelta(microseconds=505993) returned_adata_id=137078785552912


In [5]:
# Normalizing to median total counts
@logged
sc.pp.normalize_total(pbmcs)

SyntaxError: invalid syntax (5253161.py, line 3)

# HARD WAY

In [6]:
from anndata import AnnData
from copy import copy
from datetime import datetime
from functools import wraps
import inspect
from itertools import chain
from structlog import get_logger
from time import sleep
import uuid

logger = get_logger()

def logged(logged_args=None):
    """
    Params
    ------
    logged_args : list[str], optional (default: `None`)
        Names of arguments to log.
    """
    if logged_args is None:
        logged_args = []
    def logged_decorator(func):
        argnames = inspect.getfullargspec(func).args
        @wraps(func)
        def func_wrapper(*args, **kwargs):
            call_id = uuid.uuid4() # So we can always match call start with call end
            logged_params = {}

            for param, val in chain(zip(argnames, args), kwargs.items()):
                if type(val) is AnnData:
                    logged_params[param] = id(val)
                elif param in logged_args:
                    logged_params[param] = copy(val) # Probably need to consider how these values get logged

            logger.msg("call", called_func=func.__name__,
                       logged_args=logged_params, call_id=call_id)

            t0 = datetime.now()
            output = func(*args, **kwargs)
            dt = datetime.now() - t0

            call_finish_record = dict(
                called_func=func.__name__, elapsed=dt,
            )
            if type(output) is AnnData:
                call_finish_record["returned_adata_id"] = id(output)
            logger.msg("call_finish", **call_finish_record, call_id=call_id)
            return output
        return func_wrapper
    return logged_decorator

# Usage

@logged(logged_args=["x"])
def foo(adata, x, copy=True):
    sleep(0.5)
    if copy: return adata.copy()

import scanpy as sc
pbmcs = sc.datasets.pbmc68k_reduced()

foo(pbmcs, 1, copy=True)
# 2019-02-13 19:35.48 call                           call_id=UUID('f7623504-31c5-4afa-ae26-4f58fc5341a8') called_func=foo logged_args={'adata': 4476410456, 'x': 1}
# 2019-02-13 19:35.48 call_finish                    call_id=UUID('f7623504-31c5-4afa-ae26-4f58fc5341a8') called_func=foo elapsed=datetime.timedelta(microseconds=507880) returned_adata_id=5064494384

2026-02-15 12:01:20 [info     ] call                           call_id=UUID('10e8f7ba-13fb-4015-b7cd-788edbfe1f9f') called_func=foo logged_args={'adata': 140425654424912, 'x': 1}
2026-02-15 12:01:20 [info     ] call_finish                    call_id=UUID('10e8f7ba-13fb-4015-b7cd-788edbfe1f9f') called_func=foo elapsed=datetime.timedelta(microseconds=505381) returned_adata_id=140425654612496


AnnData object with n_obs × n_vars = 700 × 765
    obs: 'bulk_labels', 'n_genes', 'percent_mito', 'n_counts', 'S_score', 'G2M_score', 'phase', 'louvain'
    var: 'n_counts', 'means', 'dispersions', 'dispersions_norm', 'highly_variable'
    uns: 'bulk_labels_colors', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [18]:
import sys
import numpy as np
from anndata import AnnData
import scanpy as sc

sc.settings.verbosity = "info"
sc.settings.logfile = "/workspaces/project/analysis/py/logging.txt"
np.set_printoptions(precision=2)
adata = AnnData(
     np.array(
         [
             [3, 3, 3, 6, 6],
             [1, 1, 1, 2, 2],
             [1, 22, 1, 2, 2],
         ],
         dtype="float32",
     )
 )
adata.X


array([[ 3.,  3.,  3.,  6.,  6.],
       [ 1.,  1.,  1.,  2.,  2.],
       [ 1., 22.,  1.,  2.,  2.]], dtype=float32)

In [ ]:
adata = adata.obs[...]

X_norm = sc.pp.normalize_total(adata, target_sum=1, inplace=False)["X"]
X_norm


array([[0.14, 0.14, 0.14, 0.29, 0.29],
       [0.14, 0.14, 0.14, 0.29, 0.29],
       [0.04, 0.79, 0.04, 0.07, 0.07]], dtype=float32)

In [20]:

X_norm = sc.pp.normalize_total(
     adata,
     target_sum=1,
     exclude_highly_expressed=True,
     max_fraction=0.2,
     inplace=False,
 )["X"]


In [ ]:

omicslog.init()

1

sc.settings = omicslog.file("")

2

omicslog.file = "/dfiie/kfke/logging.txt"



omicslog(andata)



anndata.obs[[:2,:]].copy() ---x---> omicslog.filter(anndata, samples=[])

array([[ 0.5,  0.5,  0.5,  1. ,  1. ],
       [ 0.5,  0.5,  0.5,  1. ,  1. ],
       [ 0.5, 11. ,  0.5,  1. ,  1. ]], dtype=float32)